# Segmentación Semántica de Grietas en Video con YOLOv11-Seg

Este notebook implementa el pipeline de inferencia en video para la segmentación de grietas utilizando la arquitectura **YOLOv11-Seg** (*Ultralytics*), como contraparte de los notebooks de PIDNet y UNet++.

### Características Principales:
- **100% Compatible con Google Colab y Entorno Local**: Montaje automático de Drive y resolución robusta de rutas.
- **Configuración Centralizada**: Rutas y parámetros editables al inicio del notebook.
- **Máscara Semántica Unificada**: Las máscaras de instancia predichas por YOLO se fusionan (unión lógica) en una única máscara binaria de grieta, para una comparación equitativa con los modelos semánticos puros (PIDNet, UNet++).
- **Superposición Vectorizada y Telemetría HUD**: Overlay de máscara semitransparente con contornos de alta visibilidad (`cv2.drawContours`) y telemetría en tiempo real (FPS, estado y porcentaje de área de grieta).
- **Reporte Estadístico y Timeline de Daño**: Gráfico de la evolución del área de grieta a lo largo del tiempo, visualización de keyframes y reproductor HTML5 integrado en Colab.

In [ ]:
# ==== Solo para ejecutar en Colab / Local ========
try:
    from google.colab import drive
    import os
    import sys

    # 1. Montar Google Drive
    drive.mount('/content/drive')

    # 2. Definir la ruta a tu proyecto dentro de Drive
    project_path = '/content/drive/MyDrive/tp_computer_vision_ii'
    notebook_dir = os.path.join(project_path, 'src', 'yolo_seg')

    # 3. Movernos al directorio del notebook para que las rutas relativas funcionen
    os.chdir(notebook_dir)

    # 4. Agregar la ruta a sys.path
    if notebook_dir not in sys.path:
        sys.path.append(notebook_dir)

    # Instalar ultralytics si no está presente en la sesión de Colab
    try:
        import ultralytics  # noqa
    except ImportError:
        !pip install -q ultralytics

    print("Directorio actual:", os.getcwd())
    IN_COLAB = True

except Exception:
    IN_COLAB = False
    import os, sys
    current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
    if current_dir not in sys.path:
        sys.path.append(current_dir)
    print("Ejecutando en entorno Local. Directorio actual:", os.getcwd())

In [ ]:
# ==============================================================================
# 1. PARÁMETROS Y RUTAS CONFIGURABLES DE ENTRADA / SALIDA
# ==============================================================================

# ─── Rutas de Archivos (Compatibles con Colab y Local) ────────────────────────
# Pesos del modelo entrenado (best.pt). Descomentar la variante deseada:
CHECKPOINT_PATH   = "runs/yolo_seg/yolo11s_seg_small/weights/best.pt"    # Small (imgsz=512) — mejor IoU_grieta en YOLO
# CHECKPOINT_PATH = "runs/yolo_seg/yolo11n_seg_nano/weights/best.pt"     # Nano (imgsz=512) — el más rápido
# CHECKPOINT_PATH = "runs/yolo_seg/yolo11n_seg_hires/weights/best.pt"    # Nano Hi-Res (imgsz=768) — mejor IoU absoluto
# CHECKPOINT_PATH = "runs/yolo_seg/yolo11n_seg_dilated1/weights/best.pt" # Nano con máscaras dilatadas

# Video de origen a procesar
VIDEO_INPUT_PATH  = "../../datasets/videos/crack_vids.mp4"

# Video de salida con segmentación y telemetría superpuesta
VIDEO_OUTPUT_PATH = "../../datasets/videos/crack_vids_yolo11s.mp4"

# ─── Parámetros del Modelo y Segmentación ─────────────────────────────────────
MODEL_NAME        = "YOLO11s-seg"  # Etiqueta usada en el HUD
NUM_CLASSES       = 2         # 0: Fondo, 1: Grieta (comparación semántica)
IMGSZ             = 512       # Resolución de inferencia de YOLO (512 para small/nano, 768 para hires)
CONF_THRESHOLD    = 0.25      # Umbral de confianza de detección de instancias
IOU_THRESHOLD     = 0.7       # Umbral de NMS (fusión de detecciones solapadas)

# ─── Parámetros de Visualización y HUD ────────────────────────────────────────
OVERLAY_ALPHA     = 0.45      # Opacidad de la máscara de grieta (0.0 = invisible, 1.0 = opaco)
MASK_COLOR_BGR    = (0, 0, 255)    # Color de la máscara (Rojo en BGR) — idéntico a PIDNet/UNet++
DRAW_CONTOURS     = True           # Dibujar bordes/contornos vectoriales sobre la grieta
CONTOUR_COLOR_BGR = (0, 255, 255)  # Color del contorno (Amarillo en BGR)
SHOW_HUD          = True           # Mostrar panel HUD con telemetría (Frame, FPS, % Área Grieta)

# ─── Rendimiento y Aceleración ────────────────────────────────────────────────
USE_HALF          = True      # Inferencia en FP16 (media precisión) en GPU
MAX_FRAMES        = None      # Límite de frames a procesar (None = video completo)

In [ ]:
# ==============================================================================
# 2. CONFIGURACIÓN DE DEPENDENCIAS, DISPOSITIVO Y RUTAS
# ==============================================================================
import os
import sys
import time
from pathlib import Path
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from ultralytics import YOLO

# Selección de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de cómputo: {device}")
if device.type == 'cuda':
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")

def resolve_path(path_str):
    """
    Resuelve rutas de forma robusta y cross-platform tanto en Colab como en Local,
    soportando referencias relativas al directorio actual o a la raíz del proyecto.
    """
    clean_p = path_str.replace('\\', '/')
    p = Path(clean_p)
    if p.exists():
        return str(p)

    # Búsqueda en rutas relativas comunes
    search_roots = ['.', '..', '../..', 'src/yolo_seg', 'tp_computer_vision_ii']
    for root in search_roots:
        candidate = Path(root) / clean_p
        if candidate.exists():
            return str(candidate.resolve())
    return str(p)

In [ ]:
# ==============================================================================
# 3. CARGA DEL MODELO YOLOv11-SEG Y WARM-UP
# ==============================================================================

def load_yolo_model(checkpoint_path, device="cpu"):
    """
    Carga los pesos entrenados de YOLOv11-Seg (Ultralytics) y ejecuta un warm-up.
    """
    ckpt_resolved = resolve_path(checkpoint_path)
    if not os.path.exists(ckpt_resolved):
        raise FileNotFoundError(f"No se encontró el checkpoint en: {checkpoint_path} (resuelto como: {ckpt_resolved})")

    print(f"Cargando pesos desde: {ckpt_resolved}")
    model = YOLO(ckpt_resolved)
    model.to(device)

    # Warm-up pass para compilar kernels de CUDA y eliminar latencia inicial
    if device.type == 'cuda':
        dummy = np.zeros((IMGSZ, IMGSZ, 3), dtype=np.uint8)
        _ = model.predict(dummy, imgsz=IMGSZ, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
                          half=USE_HALF, device=0, verbose=False)
        torch.cuda.synchronize()
        print("Warm-up en GPU completado exitosamente.")

    return model

model = load_yolo_model(CHECKPOINT_PATH, device=device)

In [ ]:
# ==============================================================================
# 4. FUNCIONES DE PROCESAMIENTO Y OVERLAY
# ==============================================================================

def masks_to_semantic(result, out_h, out_w):
    """
    Fusiona todas las máscaras de instancia predichas por YOLO en una única
    máscara binaria semántica de grieta [out_h, out_w], reescalada a la
    resolución original del frame. Devuelve una máscara vacía si no hay detecciones.
    """
    if result.masks is None or len(result.masks) == 0:
        return np.zeros((out_h, out_w), dtype=np.uint8)
    # data: [N, mh, mw] en el espacio de la red -> unión sobre el eje de instancias
    m = result.masks.data.cpu().numpy().max(axis=0)
    m = cv2.resize(m, (out_w, out_h), interpolation=cv2.INTER_LINEAR)
    return (m > 0.5).astype(np.uint8)

def apply_crack_overlay(frame_bgr, mask_binary, alpha=0.45, mask_color_bgr=(0, 0, 255),
                        draw_contours=True, contour_color_bgr=(0, 255, 255), hud_lines=None):
    """
    Superpone la máscara de segmentación y contornos sobre el frame original
    de forma vectorizada y rápida. Idéntica a la usada en PIDNet/UNet++.
    """
    out_frame = frame_bgr.copy()
    if np.any(mask_binary):
        mask_bool = mask_binary.astype(bool)
        color_layer = np.full_like(frame_bgr, mask_color_bgr, dtype=np.uint8)

        # Blending vectorizado sobre píxeles de grieta
        out_frame[mask_bool] = cv2.addWeighted(
            frame_bgr[mask_bool], 1.0 - alpha,
            color_layer[mask_bool], alpha, 0
        )

        # Contornos perimetrales nítidos
        if draw_contours:
            contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(out_frame, contours, -1, contour_color_bgr, 1, cv2.LINE_AA)

    # Panel HUD de telemetría
    if hud_lines:
        box_w = 340
        box_h = 16 + len(hud_lines) * 26
        cv2.rectangle(out_frame, (10, 10), (10 + box_w, 10 + box_h), (20, 20, 20), -1)
        cv2.rectangle(out_frame, (10, 10), (10 + box_w, 10 + box_h), (200, 200, 200), 1)
        for i, line in enumerate(hud_lines):
            y_pos = 32 + i * 26
            cv2.putText(out_frame, line, (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 255, 255), 1, cv2.LINE_AA)

    return out_frame

In [ ]:
# ==============================================================================
# 5. MOTOR DE INFERENCIA EN VIDEO (INFERENCE ENGINE)
# ==============================================================================

video_in_resolved = resolve_path(VIDEO_INPUT_PATH)
video_out_resolved = resolve_path(VIDEO_OUTPUT_PATH)

if not os.path.exists(video_in_resolved):
    raise FileNotFoundError(f"No se encontró el video de entrada en: {video_in_resolved}")

out_dir = os.path.dirname(os.path.abspath(video_out_resolved))
if out_dir:
    os.makedirs(out_dir, exist_ok=True)

# Abrir video de entrada
cap = cv2.VideoCapture(video_in_resolved)
if not cap.isOpened():
    raise IOError(f"No se pudo abrir el archivo de video: {video_in_resolved}")

fps_in = cap.get(cv2.CAP_PROP_FPS)
if fps_in <= 0 or np.isnan(fps_in):
    fps_in = 25.0
orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if MAX_FRAMES is not None and MAX_FRAMES > 0:
    total_frames = min(total_frames, MAX_FRAMES)

# Configurar VideoWriter (misma resolución que la entrada)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(video_out_resolved, fourcc, fps_in, (orig_w, orig_h))

print(f"Resolución original: {orig_w}x{orig_h} @ {fps_in:.2f} FPS")
print(f"Inferencia YOLO:     imgsz={IMGSZ} | conf={CONF_THRESHOLD} | half={USE_HALF}")
print(f"Frames a procesar:   {total_frames}")

# Estructuras para recolección de estadísticas
timeline_stats = {
    'frame_idx': [],
    'time_sec': [],
    'crack_area_pct': [],
    'has_crack': [],
}

frame_count = 0
total_infer_time = 0.0
t_start_pipeline = time.perf_counter()
use_half_cuda = (USE_HALF and device.type == 'cuda')

pbar = tqdm(total=total_frames, desc="Segmentando Video", unit="frame")

while frame_count < total_frames:
    ret, frame = cap.read()
    if not ret:
        break

    # Inferencia de la red (medición neta en GPU)
    t_inf_start = time.perf_counter()
    result = model.predict(frame, imgsz=IMGSZ, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
                           half=use_half_cuda, device=(0 if device.type == 'cuda' else 'cpu'),
                           verbose=False)[0]
    mask_pred = masks_to_semantic(result, orig_h, orig_w)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    batch_inf_time = time.perf_counter() - t_inf_start
    total_infer_time += batch_inf_time

    # Estadísticas de severidad
    crack_pixels = int(mask_pred.sum())
    crack_pct = (crack_pixels / (orig_h * orig_w)) * 100.0
    has_crack = crack_pct > 0.01
    instant_fps = 1.0 / max(1e-5, batch_inf_time)

    hud_lines = None
    if SHOW_HUD:
        hud_lines = [
            f"{MODEL_NAME} | Frame: {frame_count + 1}/{total_frames}",
            f"FPS Inferencia: {instant_fps:.1f}",
            f"Grieta: {'DETECTADA' if has_crack else 'NO DETECTADA'}",
            f"Severidad Area: {crack_pct:.2f}%"
        ]

    frame_annotated = apply_crack_overlay(
        frame, mask_pred,
        alpha=OVERLAY_ALPHA,
        mask_color_bgr=MASK_COLOR_BGR,
        draw_contours=DRAW_CONTOURS,
        contour_color_bgr=CONTOUR_COLOR_BGR,
        hud_lines=hud_lines
    )
    writer.write(frame_annotated)

    # Registro de métricas
    timeline_stats['frame_idx'].append(frame_count)
    timeline_stats['time_sec'].append(frame_count / fps_in)
    timeline_stats['crack_area_pct'].append(crack_pct)
    timeline_stats['has_crack'].append(has_crack)

    frame_count += 1
    pbar.update(1)

pbar.close()
cap.release()
writer.release()

total_pipeline_time = time.perf_counter() - t_start_pipeline
avg_pipeline_fps = frame_count / total_pipeline_time if total_pipeline_time > 0 else 0
avg_infer_fps = frame_count / total_infer_time if total_infer_time > 0 else 0

print("\n" + "="*60)
print("              RESUMEN DE PROCESAMIENTO DE VIDEO             ")
print("="*60)
print(f" Video de entrada           : {video_in_resolved}")
print(f" Video de salida            : {video_out_resolved}")
print(f" Frames totales procesados  : {frame_count}")
print(f" Tiempo total pipeline      : {total_pipeline_time:.2f} s")
print(f" Tiempo neto inferencia GPU : {total_infer_time:.2f} s")
print(f" FPS Pipeline completo      : {avg_pipeline_fps:.2f} FPS")
print(f" FPS Inferencia neta (Red)  : {avg_infer_fps:.2f} FPS")
print("="*60)

In [ ]:
# ==============================================================================
# 6. ANÁLISIS ESTADÍSTICO Y TIMELINE DE DAÑO / SEVERIDAD
# ==============================================================================

if len(timeline_stats['frame_idx']) > 0:
    crack_pct_arr = np.array(timeline_stats['crack_area_pct'])
    frames_with_crack = sum(timeline_stats['has_crack'])
    total_f = len(crack_pct_arr)

    print("--- Estadísticas de Defectos en el Video ---")
    print(f"Frames con grieta detectada : {frames_with_crack}/{total_f} ({frames_with_crack/total_f*100:.1f}%)")
    print(f"Porcentaje de área promedio : {crack_pct_arr.mean():.3f}%")
    print(f"Porcentaje de área máximo   : {crack_pct_arr.max():.3f}% (Frame #{timeline_stats['frame_idx'][int(crack_pct_arr.argmax())]})")

    # Gráfico de línea temporal
    plt.figure(figsize=(14, 4.5))
    plt.plot(timeline_stats['time_sec'], timeline_stats['crack_area_pct'], color='crimson', lw=1.8, label='% Área Grieta')
    plt.axhline(y=crack_pct_arr.mean(), color='navy', linestyle='--', alpha=0.7, label=f'Promedio: {crack_pct_arr.mean():.2f}%')
    plt.fill_between(timeline_stats['time_sec'], timeline_stats['crack_area_pct'], color='crimson', alpha=0.15)

    plt.xlabel('Tiempo de Video (segundos)', fontsize=11)
    plt.ylabel('Superficie de Grieta (%)', fontsize=11)
    plt.title('Línea Temporal de Severidad y Detección de Grietas (YOLOv11-Seg)', fontsize=12, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ==============================================================================
# 7. VISUALIZACIÓN DE KEYFRAMES Y REPRODUCCIÓN EN NOTEBOOK / COLAB
# ==============================================================================

def sample_video_keyframes(video_path, num_samples=4):
    """
    Extrae y visualiza `num_samples` cuadros equidistantes del video resultante.
    """
    cap_out = cv2.VideoCapture(video_path)
    if not cap_out.isOpened():
        print(f"No se pudo abrir el video para extraer keyframes: {video_path}")
        return

    n_frames = int(cap_out.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, max(0, n_frames - 1), num_samples, dtype=int)

    fig, axes = plt.subplots(1, num_samples, figsize=(4.5 * num_samples, 4))
    if num_samples == 1:
        axes = [axes]

    for i, f_idx in enumerate(indices):
        cap_out.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
        ret, frame = cap_out.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            axes[i].imshow(frame_rgb)
            axes[i].set_title(f"Frame #{f_idx}", fontsize=11, fontweight='bold')
        axes[i].axis('off')

    cap_out.release()
    plt.tight_layout()
    plt.show()

# Visualizar 4 cuadros de muestra del video generado
sample_video_keyframes(video_out_resolved, num_samples=4)

# ─── Reproductor de Video Integrado (Para Colab) ──────────────────────────────
def play_video_inline(video_path, width=720):
    """
    Reproduce el video directamente dentro de la celda de Colab / Jupyter si está en formato compatible.
    """
    from IPython.display import HTML
    from base64 import b64encode

    if not os.path.exists(video_path):
        print(f"Archivo no encontrado: {video_path}")
        return

    size_mb = os.path.getsize(video_path) / (1024 * 1024)
    if size_mb <= 50:
        with open(video_path, 'rb') as f:
            mp4_data = f.read()
        data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()
        return HTML(f"""
        <video width="{width}" controls style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.15);">
            <source src="{data_url}" type="video/mp4">
            Tu navegador no soporta reproducción HTML5 directa.
        </video>
        """)
    else:
        print(f"El video pesa {size_mb:.1f} MB (demasiado grande para embed base64). Descárgalo o reprodúcelo localmente.")

# Descomentar para reproducir en Colab:
# play_video_inline(video_out_resolved)